# 📝 Task 5: Documentation & Code Review
## Báo cáo Chuyên sâu: Kiến trúc Mô hình và Pipeline
### Cuộc thi: LLM — Detect AI Generated Text (Kaggle)

> **Ngôn ngữ báo cáo:** Tiếng Việt học thuật  
> **Phạm vi:** Toàn bộ pipeline từ data preprocessing đến ensemble inference

---

## I. Tổng quan Bài toán

### 1.1 Định nghĩa

Bài toán **Phân loại văn bản do AI sinh ra** (AI-Generated Text Detection) được phát biểu như một bài toán **Phân loại nhị phân** (Binary Classification):

- **Đầu vào (Input):** Một đoạn văn bản $x \in \mathcal{X}$
- **Đầu ra (Output):** Nhãn $y \in \{0, 1\}$ với:
  - $y = 0$: Văn bản do Con người viết (Human)
  - $y = 1$: Văn bản do Mô hình Ngôn ngữ Lớn (LLM) sinh ra (AI)
- **Mục tiêu tối ưu:** Tối đa hóa chỉ số **ROC-AUC** trên Leaderboard của Kaggle

### 1.2 Thách thức

| Thách thức | Mô tả |
|------------|-------|
| **Domain Generalization** | Dataset bao gồm văn bản từ nhiều LLM khác nhau (GPT-4, Claude, Llama...) |
| **Class Imbalance** | Tỷ lệ Human/AI có thể không cân bằng |
| **Short Text** | Một số văn bản có < 100 từ, khó phân biệt |
| **Paraphrasing** | Văn bản AI được sửa bởi con người hoặc ngược lại |

## II. Kiến trúc Tổng thể Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│                    RAW TEXT INPUT                           │
└──────────────────────┬──────────────────────────────────────┘
                       │
          ┌────────────▼────────────┐
          │   TEXT CLEANING         │
          │  (Unicode, URL, Email)  │
          └────────────┬────────────┘
                       │
           ┌───────────┼────────────────────┐
           │           │                    │
    ┌──────▼──────┐ ┌──▼──────────┐  ┌──────▼──────┐
    │  TF-IDF +   │ │ TF-IDF +    │  │  DeBERTa    │
    │  Logistic   │ │ LightGBM    │  │  v3-base    │
    │ Regression  │ │             │  │  Fine-tuned │
    └──────┬──────┘ └──┬──────────┘  └──────┬──────┘
           │           │                    │
    prob_lr        prob_lgbm          prob_deberta
           │           │                    │
           └───────────┼────────────────────┘
                       │
          ┌────────────▼────────────┐
          │  WEIGHTED ENSEMBLE      │
          │  (Nelder-Mead opt.)     │
          └────────────┬────────────┘
                       │
          ┌────────────▼────────────┐
          │   FINAL PREDICTION      │
          │   P(AI) ∈ [0, 1]        │
          └─────────────────────────┘
```

## III. Kiến trúc DeBERTa-v3 — Phân tích Chi tiết

### 3.1 Tổng quan DeBERTa

**DeBERTa** (Decoding-enhanced BERT with disentangled attention) được Microsoft phát triển với hai cải tiến chính so với BERT/RoBERTa:

#### a) Disentangled Attention (Chú ý Phân ly)

Trong BERT, mỗi token được biểu diễn bởi **một vector duy nhất** kết hợp cả nội dung và vị trí. DeBERTa tách biệt thành **hai vector riêng biệt**:

$$
A_{i,j} = \underbrace{H_i W_q (H_j W_k)^T}_{\text{content-to-content}} +
           \underbrace{H_i W_q (P_{i|j} W_k)^T}_{\text{content-to-position}} +
           \underbrace{P_{j|i} W_q (H_j W_k)^T}_{\text{position-to-content}}
$$

Trong đó:
- $H_i$: Vector biểu diễn nội dung token thứ $i$
- $P_{i|j}$: Vector biểu diễn vị trí tương đối của $i$ so với $j$

**Lợi ích cho LLM Detection:** AI thường có cấu trúc câu và pattern vị trí từ khác con người, việc phân tách nội dung-vị trí giúp DeBERTa nhận diện tốt hơn.

#### b) Enhanced Mask Decoder (EMD)

DeBERTa sử dụng absolute position embedding chỉ tại bước softmax cuối (thay vì đầu vào như BERT), giúp mô hình học position tốt hơn trong quá trình pre-training.

#### c) DeBERTa-v3: Scale-invariant Fine-tuning (SiFT)

DeBERTa-v3 sử dụng kỹ thuật **RTGM (Replaced Token Detection)** từ ELECTRA và **SiFT** để cải thiện khả năng fine-tuning trên downstream tasks.

### 3.2 Cấu hình mô hình

| Thông số | Giá trị |
|----------|--------|
| Backbone | `microsoft/deberta-v3-base` |
| Hidden size | 768 |
| Attention heads | 12 |
| Layers | 12 |
| Max position embeddings | 512 |
| Vocab size | 128,100 |
| Total parameters | ~183M |
| Pooling strategy | Mean Pooling qua attention mask |
| Classifier | Linear(768, 2) với Dropout(0.1) |

## IV. Kỹ thuật Tối ưu hóa

### 4.1 Layerwise Learning Rate Decay (LLRD)

Đây là kỹ thuật quan trọng nhất trong fine-tuning Transformer. Thay vì dùng cùng một learning rate cho toàn bộ mô hình, ta gán **learning rate giảm dần** từ các layer trên xuống các layer dưới:

$$
\text{lr}_{\text{layer } l} = \text{lr}_{\text{base}} \times \alpha^{L - l}
$$

Trong đó $\alpha \in (0.8, 0.95)$ là hệ số decay, $L$ là tổng số layers.

**Lý do:** Các layer đầu của Transformer học các đặc trưng cú pháp cơ bản (đã rất tốt từ pre-training), trong khi các layer cuối học ngữ nghĩa cụ thể hơn. Dùng LR thấp ở các layer đầu giúp tránh **catastrophic forgetting**.

Trong implementation của chúng ta:
```
Backbone params: lr = 2e-5
Pool layer:      lr = 2e-5 × 5 = 1e-4
Classifier head: lr = 2e-5 × 10 = 2e-4
```

### 4.2 Gradient Accumulation

Thay vì cập nhật gradient sau mỗi batch, ta **tích lũy gradient** qua nhiều bước:

$$
g_{\text{accum}} = \frac{1}{K} \sum_{k=1}^{K} \nabla \mathcal{L}(\text{batch}_k)
$$

Với `grad_accum_steps=2` và `batch_size=16`, **effective batch size** thực tế là **32**, giúp huấn luyện ổn định hơn khi VRAM hạn chế.

### 4.3 Mixed Precision Training (AMP)

**Automatic Mixed Precision (FP16)** tính toán forward pass với `float16` và backward pass với `float32` (qua GradScaler), giúp:
- Tăng tốc ~2x trên GPU hiện đại có Tensor Core
- Giảm VRAM tiêu thụ ~50%
- Không ảnh hưởng đến độ chính xác nhờ dynamic loss scaling

### 4.4 Cosine Learning Rate Schedule với Warmup

$$
\text{lr}(t) = \begin{cases}
\text{lr}_{\max} \cdot \dfrac{t}{T_{\text{warmup}}} & t < T_{\text{warmup}} \\
\dfrac{\text{lr}_{\max}}{2}\left(1 + \cos\dfrac{\pi(t - T_{\text{warmup}})}{T_{\text{total}} - T_{\text{warmup}}}\right) & t \geq T_{\text{warmup}}
\end{cases}
$$

Phase **warmup** (10% đầu) giúp mô hình thích nghi dần với learning rate cao, tránh gradient explosion ở giai đoạn đầu fine-tuning.

### 4.5 Label Smoothing

Thay vì dùng one-hot labels cứng, ta **làm mềm phân phối nhãn**:

$$
y_{\text{smooth}} = (1 - \epsilon) \cdot y_{\text{true}} + \dfrac{\epsilon}{K}
$$

Với $\epsilon = 0.05$ và $K=2$ classes. Kỹ thuật này giúp mô hình **không over-confident**, cải thiện khả năng tổng quát hóa khi dataset có nhiễu nhãn.

## V. Chiến lược Ensemble

### 5.1 Lý do sử dụng Ensemble

**Bias-Variance Tradeoff:** Mỗi loại model có điểm mạnh khác nhau:

| Model | Điểm mạnh | Điểm yếu |
|-------|-----------|----------|
| **LR + TF-IDF** | Interpretable, nhanh, không cần GPU | Không nắm được ngữ nghĩa sâu |
| **LightGBM + TF-IDF** | Xử lý nonlinear features, robust | Cũng bị giới hạn bởi bag-of-words |
| **DeBERTa (fine-tuned)** | Hiểu ngữ cảnh sâu, state-of-the-art | Cần GPU, overfitting nếu ít data |

Ensemble kết hợp **đặc trưng thống kê từ n-grams** (bắt pattern bề mặt của AI) với **ngữ nghĩa sâu từ Transformer** (hiểu cấu trúc câu, coherence) để tạo mô hình mạnh mẽ hơn.

### 5.2 Weighted Average Ensemble

**Hàm mục tiêu tối ưu:**

$$
\mathbf{w}^* = \arg\max_{\mathbf{w}} \text{AUC}\left(\mathbf{y},\ \sum_{m=1}^{M} w_m \cdot p_m\right)
\quad \text{s.t.} \quad w_m \geq 0,\ \sum_{m=1}^{M} w_m = 1
$$

Ta sử dụng **Nelder-Mead simplex method** (không cần gradient) để tìm optimal weights, bởi vì:
1. Số chiều nhỏ (chỉ 3 tham số)
2. AUC không differentiable
3. Hội tụ nhanh với N_ITERATIONS ≈ 200-500

### 5.3 Phân tích lý thuyết

**Định lý Ensemble (Tóm tắt):** Nếu các models có correlation thấp và accuracy > 0.5, ensemble luôn tốt hơn individual models.

$$
\text{Error}_{\text{ensemble}} = \bar{b}^2 + \frac{1 - \bar{r}}{M} \bar{\sigma}^2
$$

Trong đó $\bar{b}$ là bias trung bình, $\bar{r}$ là correlation trung bình, $\bar{\sigma}^2$ là variance trung bình. Khi $\bar{r}$ thấp (models không tương quan), ensemble giảm variance đáng kể.

## VI. Code Review — Các Điểm Tối ưu Quan trọng

In [ ]:
# ════════════════════════════════════════════════════════════
#  CODE REVIEW — CHECKLIST & PHÂN TÍCH
# ════════════════════════════════════════════════════════════

review_items = [
    {
        'section': 'Task 2 — Data Pipeline',
        'items': [
            ('✅ ĐÚNG',    'TextCleaner compile regex pattern một lần duy nhất trong __init__ (tránh recompile mỗi lần gọi clean)'),
            ('✅ ĐÚNG',    'Stratified K-Fold đảm bảo tỷ lệ nhãn đồng đều giữa các fold'),
            ('✅ ĐÚNG',    'pin_memory=True khi dùng CUDA để tăng tốc CPU→GPU transfer'),
            ('✅ ĐÚNG',    'num_workers=0 trên Windows (tránh deadlock với multiprocessing.fork)'),
            ('⚠️ CẢI TIẾN', 'Nên thêm cache tokenization bằng HuggingFace datasets.map() cho dataset lớn'),
            ('⚠️ CẢI TIẾN', 'Xem xét dùng padding=\"longest\" (dynamic padding) thay vì max_length để tăng tốc'),
        ]
    },
    {
        'section': 'Task 3 — Training Loop',
        'items': [
            ('✅ ĐÚNG',    'scaler.unscale_(optimizer) trước clip_grad_norm (QUAN TRỌNG — clip trên unscaled grads)'),
            ('✅ ĐÚNG',    'optimizer.zero_grad() sau scaler.step() (tránh zero grad trước khi accumulate)'),
            ('✅ ĐÚNG',    'autocast(enabled=cfg.use_amp) — dùng enabled flag thay vì if/else'),
            ('✅ ĐÚNG',    'non_blocking=True trên .to(DEVICE) để tensor transfer song song với computation'),
            ('✅ ĐÚNG',    'torch.no_grad() decorator trên valid_epoch thay vì context manager'),
            ('⚠️ CẢI TIẾN', 'Nên thêm torch.cuda.empty_cache() sau mỗi validation epoch'),
            ('⚠️ CẢI TIẾN', 'Xem xét gradient checkpointing cho text dài >512 tokens để tiết kiệm VRAM'),
            ('❌ LỖI ĐỂ MẮT', 'logits.float() cần thiết trước khi tính metrics — FP16 logits có thể gây NaN trong softmax'),
        ]
    },
    {
        'section': 'Task 4 — Ensemble',
        'items': [
            ('✅ ĐÚNG',    'Chỉ tối ưu weights trên validation set — tránh data leakage'),
            ('✅ ĐÚNG',    'Nelder-Mead phù hợp hơn gradient methods vì AUC không differentiable'),
            ('✅ ĐÚNG',    'Khởi tạo w0 proportional to AUC — warm start giúp optim hội tụ nhanh'),
            ('⚠️ CẢI TIẾN', 'Nên dùng OOF (Out-of-Fold) predictions thay vì valid predictions để tối ưu weights'),
            ('⚠️ CẢI TIẾN', 'Thử thêm Power Average Ensemble: p = (w1*p1^α + w2*p2^α)^(1/α)'),
            ('⚠️ CẢI TIẾN', 'LightGBM nên dùng SparseMatrix để tiết kiệm memory với TF-IDF features'),
        ]
    },
    {
        'section': 'Best Practices Chung',
        'items': [
            ('✅ ĐÚNG',    'seed_everything() được gọi trước mỗi lần training'),
            ('✅ ĐÚNG',    'Checkpoint lưu epoch, model state, optimizer state và AUC'),
            ('✅ ĐÚNG',    'gc.collect() + torch.cuda.empty_cache() sau mỗi epoch'),
            ('⚠️ CẢI TIẾN', 'Thêm logging bằng wandb hoặc mlflow để theo dõi experiment'),
            ('⚠️ CẢI TIẾN', 'Thêm assert dtype kiểm tra đầu vào DataLoader'),
        ]
    }
]

for section in review_items:
    print(f"\n{'='*65}")
    print(f"  📌 {section['section']}")
    print(f"{'='*65}")
    for tag, msg in section['items']:
        print(f"  {tag:<15} | {msg}")

## VII. Fix các lỗi tối ưu hóa thường gặp

In [ ]:
# ── Lỗi 1: Thiếu .float() sau logits FP16 ────────────────────────────────────
print('=== LỖI 1: FP16 logits gây NaN trong softmax ===')
print('''
  # ❌ SAI — có thể gây NaN nếu dùng AMP
  probs = torch.softmax(logits, dim=-1)

  # ✅ ĐÚNG — cast sang float32 trước
  probs = torch.softmax(logits.float(), dim=-1)
''')

# ── Lỗi 2: GradScaler unscale thứ tự sai ─────────────────────────────────────
print('=== LỖI 2: Thứ tự unscale/clip/step sai ===')
print('''
  # ❌ SAI — clip gradient đã scale → sai magnitude
  scaler.scale(loss).backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
  scaler.step(optimizer)

  # ✅ ĐÚNG — phải unscale trước khi clip
  scaler.scale(loss).backward()
  scaler.unscale_(optimizer)  # ← PHẢI CÓ
  torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
  scaler.step(optimizer)
  scaler.update()
''')

# ── Lỗi 3: Dynamic padding — tối ưu tốc độ ───────────────────────────────────
print('=== TỐI ƯU 3: Dynamic Padding thay vì Max-Length Padding ===')
print('''
  # Cách cũ (padding='max_length') — mọi batch đều dài 512 tokens
  # wastes compute trên padding tokens

  # Cách tối ưu: padding trong collate_fn theo batch-max-length
  def smart_collate_fn(batch):
      max_len = max(b["attention_mask"].sum().item() for b in batch)
      return {
          "input_ids":      torch.stack([b["input_ids"][:max_len]      for b in batch]),
          "attention_mask": torch.stack([b["attention_mask"][:max_len] for b in batch]),
          "labels":         torch.stack([b["labels"]                   for b in batch]),
      }
  # → Tăng tốc ~20-40% khi dataset có nhiều văn bản ngắn
''')

# ── Tối ưu 4: Gradient Checkpointing ─────────────────────────────────────────
print('=== TỐI ƯU 4: Gradient Checkpointing — tiết kiệm VRAM ===')
print('''
  # Khi VRAM hạn chế mà muốn max_length > 512:
  model.backbone.gradient_checkpointing_enable()
  # → Giảm VRAM ~60% với cost tăng ~30% thời gian backward
''')

## VIII. Chiến lược cải thiện Score trên Kaggle

In [ ]:
improvements = [
    ('🥇 High Impact', [
        '5-Fold Cross Validation: huấn luyện 5 models, ensemble OOF predictions → AUC tăng ~0.005-0.01',
        'Dùng deberta-v3-large thay cho deberta-v3-base nếu VRAM cho phép (AUC tăng ~0.003-0.008)',
        'Pseudo-labeling: dự đoán test set với confidence cao → thêm vào training data → retrain',
        'Data Augmentation: Back-translation, Sentence Shuffle để tăng dữ liệu human',
    ]),
    ('🥈 Medium Impact', [
        'Multi-model Ensemble: thêm ELECTRA, RoBERTa-large vào ensemble',
        'Feature Engineering cho LightGBM: perplexity từ pre-trained GPT-2 làm feature',
        'Post-processing: điều chỉnh threshold tối ưu theo F1 thay vì 0.5',
        'Stochastic Weight Averaging (SWA): average weights của các checkpoint cuối',
    ]),
    ('🥉 Low Impact / Experimental', [
        'LightGBM Feature: avg n-gram perplexity theo sentence vs tổng thể',
        'Custom Pre-training: pre-train thêm trên domain-specific corpus',
        'Adversarial Training (FGSM/PGD): cải thiện robustness',
        'Calibration: Platt scaling để cải thiện probability calibration',
    ]),
]

print('🚀 CHIẾN LƯỢC CẢI THIỆN SCORE TRÊN KAGGLE')
print('='*65)
for tier, items in improvements:
    print(f'\n  {tier}')
    print('  ' + '─'*60)
    for item in items:
        print(f'  • {item}')
print('\n' + '='*65)

## IX. Tổng kết & Thuật ngữ

In [ ]:
glossary = {
    'ROC-AUC': 'Receiver Operating Characteristic — Area Under Curve. Đo khả năng phân biệt nhãn 0/1 với mọi ngưỡng quyết định. AUC=1 là hoàn hảo, AUC=0.5 là ngẫu nhiên.',
    'Tokenization': 'Quá trình phân tách văn bản thành các đơn vị con (subwords/tokens) để mô hình có thể xử lý. DeBERTa dùng SentencePiece BPE tokenizer.',
    'Attention Mask': 'Ma trận nhị phân [1=token thật, 0=padding] để mô hình không chú ý vào padding tokens.',
    'Mean Pooling': 'Tính trung bình có trọng số của hidden states theo attention mask. Tốt hơn CLS pooling cho classification vì đại diện toàn bộ chuỗi.',
    'Type-Token Ratio (TTR)': 'Tỷ lệ số từ duy nhất / tổng số từ. AI thường có TTR thấp hơn human do xu hướng lặp pattern.',
    'Disentangled Attention': 'Kỹ thuật tách content vector và position vector trong self-attention của DeBERTa, cải thiện khả năng học tương quan nội dung-vị trí.',
    'Gradient Accumulation': 'Gộp gradient từ K mini-batches trước khi cập nhật, tạo effective batch size lớn hơn khi VRAM hạn chế.',
    'Mixed Precision (AMP)': 'Dùng float16 cho forward/backward (nhanh hơn), float32 cho optimizer update (chính xác hơn), với GradScaler để tránh underflow.',
    'Early Stopping': 'Dừng training khi metric validation không cải thiện sau N epochs (patience), để tránh overfitting.',
    'Out-of-Fold (OOF)': 'Predictions trên validation set thu được từ K-Fold CV, đảm bảo mọi mẫu đều là dự đoán unseen khi tối ưu ensemble weights.',
    'Label Smoothing': 'Làm mềm one-hot labels sang phân phối mịn hơn để giảm overconfidence của mô hình.',
    'LLRD': 'Layerwise Learning Rate Decay — gán LR thấp dần từ classifier head xuống các attention layer đầu để tránh catastrophic forgetting.',
}

print('📚 BẢNG THUẬT NGỮ CHUYÊN NGÀNH')
print('='*70)
for term, definition in glossary.items():
    print(f'\n  📌 {term}')
    print(f'     {definition}')
print('\n' + '='*70)
print('\n✅ Task 5 — Documentation & Code Review hoàn tất!')